# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [291]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
import seaborn as sns
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt


In [292]:
SEED = 900

In [293]:
target = "daily_orders"

In [294]:
sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

In [295]:
tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)

In [296]:
def calculate_operational_score(y_true, p10, p50, p90):
    error_10 = np.where(y_true >= p10, 0.1 * (y_true - p10), 0.9 * (p10 - y_true))
    error_50 = np.where(y_true >= p50, 0.5 * (y_true - p50), 0.5 * (p50 - y_true))
    error_90 = np.where(y_true >= p90, 0.9 * (y_true - p90), 0.1 * (p90 - y_true))
    
    total_score = np.mean(error_10 + error_50 + error_90)
    return total_score


In [297]:
ids = te.id
tr, te = tr.drop(columns="id"), te.drop(columns="id")

In [298]:
cat_cols = []
for col in tr.columns:
    if tr[col].nunique() < 9:
        cat_cols.append(col)

In [299]:
tr['a'] = tr["dist_to_center_km"] / (tr["residential_units_500m"]+1e-9)
te['a'] = te["dist_to_center_km"] / (te["residential_units_500m"]+1e-9)


In [300]:
tr['b'] = tr["dist_to_bus_stop_m"] / (tr["commercial_poi_count_500m"]+1e-9)
te['b'] = te["dist_to_bus_stop_m"] / (te["commercial_poi_count_500m"]+1e-9)


In [301]:
corrs = tr.select_dtypes(include="number").corr()

In [302]:
tr[target] = np.log1p(tr[target])

In [303]:
X, y = tr.drop(columns=target), tr[target]
X_te = te

In [304]:
skf = KFold(n_splits=5, shuffle=True, random_state=SEED)
y_og = np.expm1(y) 
models = []
oof_preds = np.zeros((len(X), 3)) 
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(
        loss_function='MultiQuantile:alpha=0.1,0.5,0.9',
        iterations=3000,
        learning_rate=0.02,
        early_stopping_rounds=50,
        random_seed=SEED,
        verbose=0,
        cat_features=cat_cols,
        depth=6
    )
    

    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    
    preds_val_log = model.predict(X_val)    
    preds_val_original = np.expm1(preds_val_log)
    oof_preds[val_idx] = preds_val_original
    
    p10_val = preds_val_original[:, 0]
    p50_val = preds_val_original[:, 1]
    p90_val = preds_val_original[:, 2]
    
    y_val_true_original = y_og.iloc[val_idx].values
    
    score = calculate_operational_score(y_val_true_original, p10_val, p50_val, p90_val)
    fold_scores.append(score)
    models.append(model)
    print(f"Fold {fold} - Operational Score: {score:.4f}")



Fold 1 - Operational Score: 12.8152
Fold 2 - Operational Score: 13.8080
Fold 3 - Operational Score: 14.1238
Fold 4 - Operational Score: 12.8860
Fold 5 - Operational Score: 12.8231


In [305]:
overall_score = calculate_operational_score(y_og.values, oof_preds[:, 0], oof_preds[:, 1], oof_preds[:, 2])
print(f"\nMean Fold Score: {np.mean(fold_scores):.4f}")
print(f"Overall OOF Operational Score: {overall_score:.4f}")



Mean Fold Score: 13.2912
Overall OOF Operational Score: 13.2912


In [306]:
test_preds_list = []

for model in models:
    preds_log = model.predict(X_te)
    preds_original = np.expm1(preds_log)
    test_preds_list.append(preds_original)


In [307]:
final_test_preds = np.mean(test_preds_list, axis=0)

In [309]:
final_test_preds = np.sort(final_test_preds, axis=1)

In [310]:
df_submission = pd.DataFrame(final_test_preds, columns=[
    'predicted_daily_orders_p10', 
    'predicted_daily_orders_p50', 
    'predicted_daily_orders_p90'
], index=X_te.index)

In [311]:
mask = ((~temp["c1"]) & (~temp["c2"])).reindex(df_submission.index).fillna(False).to_numpy()

In [314]:
df_ids = pd.DataFrame(data=ids, columns=["id"])

In [315]:
df_submission = pd.concat([df_ids, df_submission], axis=1)

In [316]:
df_submission.to_csv("submission.csv", index=False)

In [317]:
cv = cv_json("cb_cv_fe(ab)", [f / 100 for f in fold_scores])
print(json.dumps(cv))

{"name": "cb_cv_fe(ab)", "metric": "quantile_quality_x100", "folds": [12.82, 13.81, 14.12, 12.89, 12.82], "cv": 13.29, "std": 0.56}
